In [ ]:
# 1. Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

# --- CONFIGURATION ---
ZIP_PATH = '/content/drive/MyDrive/FYP_Project/Testing_Benchmarks.zip'
EXTRACT_PATH = '/content/dataset/Testing_Benchmarks'

if not os.path.exists(EXTRACT_PATH):
    print(f"📂 Unzipping Benchmarks...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print(f"✅ Extracted to: {EXTRACT_PATH}")
else:
    print(f"✅ Data already exists at {EXTRACT_PATH}")

# Define the 3 Paths for the loops
BENCHMARK_SETS = [
    os.path.join(EXTRACT_PATH, "Benchmark_Set_1"),
    os.path.join(EXTRACT_PATH, "Benchmark_Set_2"),
    os.path.join(EXTRACT_PATH, "Benchmark_Set_3")
]

📂 Unzipping Benchmarks...
✅ Extracted to: /content/dataset/Testing_Benchmarks


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import pandas as pd

# --- CONFIGURATION ---
MODEL_ARCH = "ResNet101"
BASE_MODEL_DIR = "/content/drive/MyDrive/FYP_Project/Models/ResNet101"
OUTPUT_ROOT = "/content/drive/MyDrive/FYP_Project/Final_Results/ResNet101"
BENCHMARK_ROOT = "/content/dataset/Testing_Benchmarks/Testing_Benchmarks"
SETS = ["Benchmark_Set_1", "Benchmark_Set_2", "Benchmark_Set_3"]

# Define Configs
CONFIGS = [
    {"name": "ConfigA",         "path": f"{BASE_MODEL_DIR}/Run_01_ConfigA/ResNet101_Run_01_ConfigA_model.pth"},
    {"name": "ConfigB_Batch32", "path": f"{BASE_MODEL_DIR}/Run_02_ConfigB_Batch32/ResNet101_Run_02_ConfigB_Batch32_model.pth"},
    {"name": "ConfigC_SGD",     "path": f"{BASE_MODEL_DIR}/Run_03_ConfigC_SGD/ResNet101_Run_03_ConfigC_SGD_model.pth"},
    {"name": "ConfigD_LowLR",   "path": f"{BASE_MODEL_DIR}/Run_04_ConfigD_LR_Low/ResNet101_Run_04_ConfigD_LR_Low_model.pth"}
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- GRAD-CAM CLASS (Your Style) ---
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None; self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)
    def save_activation(self, module, input, output): self.activations = output
    def save_gradient(self, module, grad_input, grad_output): self.gradients = grad_output[0]
    def __call__(self, x):
        output = self.model(x); idx = torch.argmax(output, dim=1)
        self.model.zero_grad(); output[0, idx].backward()
        grads = self.gradients[0].cpu().data.numpy()
        acts = self.activations[0].cpu().data.numpy()
        weights = np.mean(grads, axis=(1, 2))
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights): cam += w * acts[i]
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam - np.min(cam); cam = cam / np.max(cam)
        return cam, idx

# --- PLOTTING HELPER ---
def save_gradcam(img_tensor, heatmap, true_label, pred_label, save_path):
    # Un-normalize
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    # Heatmap Style (Jet)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    heatmap_colored = heatmap_colored[..., ::-1] # BGR to RGB

    # Superimpose
    superimposed = 0.6 * img + 0.4 * heatmap_colored
    superimposed = superimposed / np.max(superimposed)

    plt.figure(figsize=(5, 5))
    plt.imshow(superimposed)
    plt.title(f"True: {true_label} | Pred: {pred_label}")
    plt.axis('off')
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# --- MAIN LOOP ---
results = []
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("🚀 Starting ResNet Evaluation...")

for set_name in SETS:
    test_path = os.path.join(BENCHMARK_ROOT, set_name)
    if not os.path.exists(test_path): continue

    dataset = datasets.ImageFolder(test_path, transform=data_transforms)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

    # Create a small loader for visualization (Shuffle=True to see random samples)
    viz_loader = DataLoader(dataset, batch_size=1, shuffle=True)

    print(f"\n📂 Processing {set_name}...")

    for config in CONFIGS:
        print(f"   ⚙️ {config['name']}...", end=" ")

        # 1. Structure Folders
        save_dir = os.path.join(OUTPUT_ROOT, set_name, config['name'])
        os.makedirs(save_dir, exist_ok=True)

        # 2. Load Model
        model = models.resnet101(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 2)
        try:
            model.load_state_dict(torch.load(config['path'], map_location=DEVICE))
        except:
            print("❌ Model file missing."); continue
        model = model.to(DEVICE); model.eval()

        # 3. Metrics
        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs[:, 0].cpu().numpy())

        # Calculate Stats (Fake=0, Real=1 in ImageFolder, usually we invert for medical recall)
        # Assuming 0=Fake, 1=Real. We map Fake->1 for positive class stats.
        bin_labels = [1 if x==0 else 0 for x in all_labels]
        bin_preds  = [1 if x==0 else 0 for x in all_preds]

        acc = accuracy_score(bin_labels, bin_preds)
        tn, fp, fn, tp = confusion_matrix(bin_labels, bin_preds).ravel()
        spec = tn / (tn + fp) if (tn+fp)>0 else 0
        try: auc = roc_auc_score(bin_labels, all_probs)
        except: auc = 0.5

        print(f"✅ Acc: {acc:.4f} | Spec: {spec:.4f}")
        results.append({"Dataset": set_name, "Model": MODEL_ARCH, "Config": config['name'], "Acc": acc, "Spec": spec, "AUC": auc})

        # 4. Generate Grad-CAM (Save 4 samples)
        grad_cam = GradCAM(model, model.layer4[-1])
        count = 0
        for img, label in viz_loader:
            if count >= 4: break
            img = img.to(DEVICE)
            heatmap, pred_idx = grad_cam(img)

            lbl_str = "Real" if label.item()==1 else "Fake"
            pred_str = "Real" if pred_idx.item()==1 else "Fake"

            save_path = os.path.join(save_dir, f"GradCAM_{count}_{lbl_str}.png")
            save_gradcam(img[0], heatmap, lbl_str, pred_str, save_path)
            count += 1

# Save Master CSV
pd.DataFrame(results).to_csv(os.path.join(OUTPUT_ROOT, "ResNet101_Final_Results.csv"), index=False)
print("\n🎉 ResNet Analysis Complete.")

🚀 Starting ResNet Evaluation...

📂 Processing Benchmark_Set_1...
   ⚙️ ConfigA... ✅ Acc: 0.9770 | Spec: 0.9850
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9868 | Spec: 1.0000
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9835 | Spec: 0.9825
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9858 | Spec: 0.9865

📂 Processing Benchmark_Set_2...
   ⚙️ ConfigA... ✅ Acc: 0.9775 | Spec: 0.9885
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9882 | Spec: 1.0000
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9862 | Spec: 0.9830
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9850 | Spec: 0.9860

📂 Processing Benchmark_Set_3...
   ⚙️ ConfigA... ✅ Acc: 0.9758 | Spec: 0.9850
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9892 | Spec: 1.0000
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9868 | Spec: 0.9835
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9855 | Spec: 0.9850

🎉 ResNet Analysis Complete.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import pandas as pd

# --- CONFIGURATION ---
MODEL_ARCH = "EfficientNetV2_S"
BASE_MODEL_DIR = "/content/drive/MyDrive/FYP_Project/Models/EfficientNetV2_S"
OUTPUT_ROOT = "/content/drive/MyDrive/FYP_Project/Final_Results/EfficientNet"
BENCHMARK_ROOT = "/content/dataset/Testing_Benchmarks/Testing_Benchmarks"
SETS = ["Benchmark_Set_1", "Benchmark_Set_2", "Benchmark_Set_3"]

CONFIGS = [
    {"name": "ConfigA",         "path": f"{BASE_MODEL_DIR}/Run_01_ConfigA/EfficientNetV2_S_Run_01_ConfigA_model.pth"},
    {"name": "ConfigB_Batch32", "path": f"{BASE_MODEL_DIR}/Run_02_ConfigB_Batch32/EfficientNetV2_S_Run_02_ConfigB_Batch32_model.pth"},
    {"name": "ConfigC_SGD",     "path": f"{BASE_MODEL_DIR}/Run_03_ConfigC_SGD/EfficientNetV2_S_Run_03_ConfigC_SGD_model.pth"},
    {"name": "ConfigD_LowLR",   "path": f"{BASE_MODEL_DIR}/Run_04_ConfigD_LR_Low/EfficientNetV2_S_Run_04_ConfigD_LR_Low_model.pth"}
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- GRAD-CAM (EfficientNet Target) ---
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.target_layer = target_layer
        self.gradients = None; self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)
    def save_activation(self, module, input, output): self.activations = output
    def save_gradient(self, module, grad_input, grad_output): self.gradients = grad_output[0]
    def __call__(self, x):
        output = self.model(x); idx = torch.argmax(output, dim=1)
        self.model.zero_grad(); output[0, idx].backward()
        grads = self.gradients[0].cpu().data.numpy()
        acts = self.activations[0].cpu().data.numpy()
        weights = np.mean(grads, axis=(1, 2))
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights): cam += w * acts[i]
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam - np.min(cam); cam = cam / np.max(cam)
        return cam, idx

def save_gradcam(img_tensor, heatmap, true_label, pred_label, save_path):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    heatmap_colored = heatmap_colored[..., ::-1]
    superimposed = 0.6 * img + 0.4 * heatmap_colored
    superimposed = superimposed / np.max(superimposed)
    plt.figure(figsize=(5, 5)); plt.imshow(superimposed); plt.axis('off')
    plt.title(f"True: {true_label} | Pred: {pred_label}")
    plt.savefig(save_path, bbox_inches='tight'); plt.close()

# --- MAIN LOOP ---
results = []
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("🚀 Starting EfficientNet Evaluation...")

for set_name in SETS:
    test_path = os.path.join(BENCHMARK_ROOT, set_name)
    if not os.path.exists(test_path): continue

    dataset = datasets.ImageFolder(test_path, transform=data_transforms)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
    viz_loader = DataLoader(dataset, batch_size=1, shuffle=True)

    print(f"\n📂 Processing {set_name}...")

    for config in CONFIGS:
        print(f"   ⚙️ {config['name']}...", end=" ")
        save_dir = os.path.join(OUTPUT_ROOT, set_name, config['name'])
        os.makedirs(save_dir, exist_ok=True)

        model = models.efficientnet_v2_s(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
        try:
            model.load_state_dict(torch.load(config['path'], map_location=DEVICE))
        except:
             # Fallback
            alt_path = f"{BASE_MODEL_DIR}/{config['name']}_model.pth"
            if os.path.exists(alt_path): model.load_state_dict(torch.load(alt_path, map_location=DEVICE))
            else: print("❌ Model file missing."); continue
        model = model.to(DEVICE); model.eval()

        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs[:, 0].cpu().numpy())

        bin_labels = [1 if x==0 else 0 for x in all_labels]
        bin_preds  = [1 if x==0 else 0 for x in all_preds]
        acc = accuracy_score(bin_labels, bin_preds)
        tn, fp, fn, tp = confusion_matrix(bin_labels, bin_preds).ravel()
        spec = tn / (tn + fp) if (tn+fp)>0 else 0
        try: auc = roc_auc_score(bin_labels, all_probs)
        except: auc = 0.5
        print(f"✅ Acc: {acc:.4f} | Spec: {spec:.4f}")
        results.append({"Dataset": set_name, "Model": MODEL_ARCH, "Config": config['name'], "Acc": acc, "Spec": spec, "AUC": auc})

        grad_cam = GradCAM(model, model.features[-1])
        count = 0
        for img, label in viz_loader:
            if count >= 4: break
            img = img.to(DEVICE)
            heatmap, pred_idx = grad_cam(img)
            lbl_str = "Real" if label.item()==1 else "Fake"
            pred_str = "Real" if pred_idx.item()==1 else "Fake"
            save_gradcam(img[0], heatmap, lbl_str, pred_str, os.path.join(save_dir, f"GradCAM_{count}_{lbl_str}.png"))
            count += 1

pd.DataFrame(results).to_csv(os.path.join(OUTPUT_ROOT, "EfficientNet_Final_Results.csv"), index=False)
print("\n🎉 EfficientNet Analysis Complete.")

🚀 Starting EfficientNet Evaluation...

📂 Processing Benchmark_Set_1...
   ⚙️ ConfigA... ✅ Acc: 0.9912 | Spec: 0.9995
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9862 | Spec: 0.9825
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9882 | Spec: 0.9965
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9915 | Spec: 0.9955

📂 Processing Benchmark_Set_2...
   ⚙️ ConfigA... ✅ Acc: 0.9910 | Spec: 1.0000
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9850 | Spec: 0.9810
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9915 | Spec: 0.9980
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9928 | Spec: 0.9990

📂 Processing Benchmark_Set_3...
   ⚙️ ConfigA... ✅ Acc: 0.9908 | Spec: 1.0000
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9850 | Spec: 0.9780
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9915 | Spec: 0.9995
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9938 | Spec: 0.9995

🎉 EfficientNet Analysis Complete.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import pandas as pd

# --- CONFIGURATION ---
MODEL_ARCH = "ViT_B_16"
BASE_MODEL_DIR = "/content/drive/MyDrive/FYP_Project/Models/ViT_B_16"
OUTPUT_ROOT = "/content/drive/MyDrive/FYP_Project/Final_Results/ViT"
BENCHMARK_ROOT = "/content/dataset/Testing_Benchmarks/Testing_Benchmarks"
SETS = ["Benchmark_Set_1", "Benchmark_Set_2", "Benchmark_Set_3"]

CONFIGS = [
    {"name": "ConfigA",         "path": f"{BASE_MODEL_DIR}/Run_01_ConfigA/ViT_B_16_Run_01_ConfigA_model.pth"},
    {"name": "ConfigB_Batch32", "path": f"{BASE_MODEL_DIR}/Run_02_ConfigB_Batch32/ViT_B_16_Run_02_ConfigB_Batch32_model.pth"},
    {"name": "ConfigC_SGD",     "path": f"{BASE_MODEL_DIR}/Run_03_ConfigC_SGD/ViT_B_16_Run_03_ConfigC_SGD_model.pth"},
    {"name": "ConfigD_LowLR",   "path": f"{BASE_MODEL_DIR}/Run_04_ConfigD_LR_Low/ViT_B_16_Run_04_ConfigD_LR_Low_model.pth"}
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ATTENTION MAP HELPER (Styled like GradCAM) ---
def get_attention_map(model, x):
    # Get class token attention from last layer
    x_input = model._process_input(x)
    n = x_input.shape[0]
    batch_class_token = model.class_token.expand(n, -1, -1)
    x_input = torch.cat([batch_class_token, x_input], dim=1)
    x_input = model.encoder.pos_embedding + x_input
    x_input = model.encoder.dropout(x_input)

    for layer in model.encoder.layers:
        x_input = layer(x_input)

    # Norm of the patch tokens (1 to 196)
    patches = x_input[0, 1:, :] # [196, 768]
    heatmap = patches.norm(dim=1).reshape(14, 14).detach().cpu().numpy()

    # Resize to 224x224 and Normalize
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = (heatmap - np.min(heatmap)) / np.max(heatmap)

    # Get Prediction
    logits = model.heads(x_input[:, 0]) # CLS token output
    pred_idx = torch.argmax(logits, dim=1)
    return heatmap, pred_idx

def save_gradcam_style(img_tensor, heatmap, true_label, pred_label, save_path):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)

    # Same Jet Style as ResNet/EfficientNet
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = np.float32(heatmap_colored) / 255
    heatmap_colored = heatmap_colored[..., ::-1]

    superimposed = 0.6 * img + 0.4 * heatmap_colored
    superimposed = superimposed / np.max(superimposed)

    plt.figure(figsize=(5, 5)); plt.imshow(superimposed); plt.axis('off')
    plt.title(f"True: {true_label} | Pred: {pred_label}")
    plt.savefig(save_path, bbox_inches='tight'); plt.close()

# --- MAIN LOOP ---
results = []
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("🚀 Starting ViT Evaluation...")

for set_name in SETS:
    test_path = os.path.join(BENCHMARK_ROOT, set_name)
    if not os.path.exists(test_path): continue

    dataset = datasets.ImageFolder(test_path, transform=data_transforms)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
    viz_loader = DataLoader(dataset, batch_size=1, shuffle=True)

    print(f"\n📂 Processing {set_name}...")

    for config in CONFIGS:
        print(f"   ⚙️ {config['name']}...", end=" ")
        save_dir = os.path.join(OUTPUT_ROOT, set_name, config['name'])
        os.makedirs(save_dir, exist_ok=True)

        model = models.vit_b_16(weights=None)
        model.heads.head = nn.Linear(model.heads.head.in_features, 2)
        try:
            model.load_state_dict(torch.load(config['path'], map_location=DEVICE))
        except:
             # Fallback
            alt_path = f"{BASE_MODEL_DIR}/{config['name']}_model.pth"
            if os.path.exists(alt_path): model.load_state_dict(torch.load(alt_path, map_location=DEVICE))
            else: print("❌ Model file missing."); continue
        model = model.to(DEVICE); model.eval()

        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for inputs, labels in dataloader:
                inputs = inputs.to(DEVICE)
                outputs = model(inputs)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs[:, 0].cpu().numpy())

        bin_labels = [1 if x==0 else 0 for x in all_labels]
        bin_preds  = [1 if x==0 else 0 for x in all_preds]
        acc = accuracy_score(bin_labels, bin_preds)
        tn, fp, fn, tp = confusion_matrix(bin_labels, bin_preds).ravel()
        spec = tn / (tn + fp) if (tn+fp)>0 else 0
        try: auc = roc_auc_score(bin_labels, all_probs)
        except: auc = 0.5
        print(f"✅ Acc: {acc:.4f} | Spec: {spec:.4f}")
        results.append({"Dataset": set_name, "Model": MODEL_ARCH, "Config": config['name'], "Acc": acc, "Spec": spec, "AUC": auc})

        count = 0
        for img, label in viz_loader:
            if count >= 4: break
            img = img.to(DEVICE)
            heatmap, pred_idx = get_attention_map(model, img)
            lbl_str = "Real" if label.item()==1 else "Fake"
            pred_str = "Real" if pred_idx.item()==1 else "Fake"
            save_gradcam_style(img[0], heatmap, lbl_str, pred_str, os.path.join(save_dir, f"Attention_{count}_{lbl_str}.png"))
            count += 1

pd.DataFrame(results).to_csv(os.path.join(OUTPUT_ROOT, "ViT_Final_Results.csv"), index=False)
print("\n🎉 ViT Analysis Complete.")

🚀 Starting ViT Evaluation...

📂 Processing Benchmark_Set_1...
   ⚙️ ConfigA... ✅ Acc: 0.9625 | Spec: 0.9525
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9812 | Spec: 0.9865
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9748 | Spec: 0.9850
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9740 | Spec: 0.9960

📂 Processing Benchmark_Set_2...
   ⚙️ ConfigA... ✅ Acc: 0.9620 | Spec: 0.9515
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9832 | Spec: 0.9910
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9698 | Spec: 0.9845
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9705 | Spec: 0.9970

📂 Processing Benchmark_Set_3...
   ⚙️ ConfigA... ✅ Acc: 0.9663 | Spec: 0.9585
   ⚙️ ConfigB_Batch32... ✅ Acc: 0.9845 | Spec: 0.9900
   ⚙️ ConfigC_SGD... ✅ Acc: 0.9742 | Spec: 0.9875
   ⚙️ ConfigD_LowLR... ✅ Acc: 0.9768 | Spec: 0.9980

🎉 ViT Analysis Complete.
